# The scikit-learn API, one call at a time

Every notebook of lessons 1 to 3 calls the same handful of methods: `fit`,
`transform`, `fit_transform`, `predict`, `predict_proba`, `score`. In the lesson
notebooks they run on hundreds of rows, so what each call does is hidden behind
the size of the data. Here each one runs on four to ten rows, small enough that
every number can be read and checked by hand.

**How to use this notebook.** Every section is one *atom*: one idea, one call.
Before each code cell that teaches something there is a question under the
heading *Your prediction, before you run the next cell*. Write your answer down,
in your head or in a markdown cell, and only then run the cell. A prediction that
turns out wrong is the most useful thing in the notebook: it marks the exact place
where your picture of the library differs from what the library does.

This material is **not examinable**. It repeats, at a smaller scale, what the
lessons already taught, and it points back to the lesson where each idea first
appeared. The guide `consolidation_guide.md` lists every atom with its source.

## Setup

In [1]:
# Everything used here ships with the course image:
# pip install numpy==1.26.4 pandas==2.1.1 scikit-learn==1.3.1

import numpy as np
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import accuracy_score, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

np.set_printoptions(precision=3, suppress=True)

## 1. Estimators learn in `fit`, and keep what they learned

Every scikit-learn object is an **estimator**: you create it with its settings, then
call `fit` on data, and `fit` stores what it learned in attributes whose names end with
an underscore (`mean_`, `statistics_`, `coef_`). The underscore is the library's
convention for "this exists only after `fit`". An estimator that changes data, like
a scaler or an imputer, is called a **transformer**; one that makes predictions, like
a regression, is called a **predictor** (or simply a *model*).

*From:* lesson 1, slide *Step 5: pipeline, not two steps*; lesson 2, slide *What to do about a gap*.

In [2]:
# Four customers' monthly charges, in euros. One column, so the shape is (4, 1).
charges = np.array([[20.0], [40.0], [60.0], [80.0]])
charges

array([[20.],
       [40.],
       [60.],
       [80.]])

### Your prediction, before you run the next cell

Before `fit` is called, does the scaler already have an attribute `mean_`?
And after `fit`, what number will `mean_` hold?

In [3]:
scaler = StandardScaler()          # only the settings so far: nothing learned yet
print("before fit, has mean_? ", hasattr(scaler, "mean_"))

scaler.fit(charges)                # learn the mean and the standard deviation
print("after fit,  has mean_? ", hasattr(scaler, "mean_"))
print("mean_  =", scaler.mean_)
print("scale_ =", scaler.scale_, "  (the standard deviation)")

before fit, has mean_?  False
after fit,  has mean_?  True
mean_  = [50.]
scale_ = [22.361]   (the standard deviation)


**What to look at.** Before `fit` the attribute does not exist: creating the object
learned nothing. After `fit`, `mean_` is 50 and `scale_` is 22.36, the square root of
the average squared distance from 50, which is (900 + 100 + 100 + 900) / 4 = 500.
scikit-learn divides by $m$, the number of rows, not by $m - 1$.

### Your prediction, before you run the next cell

An imputer and a linear regression are estimators too. Which attribute
will the imputer fill with what it learned, and which two will the regression fill?

In [4]:
ages = np.array([[30.0], [np.nan], [50.0], [40.0]])    # one age is missing
imputer = SimpleImputer(strategy="mean").fit(ages)
print("imputer.statistics_ =", imputer.statistics_, "  (the mean of 30, 50, 40)")

area = np.array([[80.0], [120.0], [200.0]])                # square metres
price = np.array([240_000.0, 320_000.0, 540_000.0])        # euros
regression = LinearRegression().fit(area, price)
print("regression.coef_      =", regression.coef_.round(1), "  (euros per square metre)")
print("regression.intercept_ =", round(regression.intercept_, 1))

imputer.statistics_ = [40.]   (the mean of 30, 50, 40)
regression.coef_      = [2535.7]   (euros per square metre)
regression.intercept_ = 28571.4


**What to look at.** The imputer learned one number per column, the mean of the
ages it could see: 40. The regression learned a slope (`coef_`, about 2,536 euros per
square metre) and an intercept (`intercept_`, about 28,600 euros). These are the three
houses of lesson 3's slide *The same equation, by hand*, and the numbers are the ones
on that slide.

**Try changing.** add a fifth charge of 1,000 euros to `charges` and look at how far
`mean_` and `scale_` move.

## 2. `transform` applies what `fit` learned; `fit_transform` does both

`transform` uses the numbers stored by `fit` and changes the data with them.
`fit_transform(X)` is a shortcut for `fit(X)` followed by `transform(X)` on the same
rows. That shortcut is right on the training set and wrong on the test set: the
test set must be changed with the numbers learned from the training set, so on the
test set you call `transform` only.

*From:* lesson 2, slide *StandardScaler vs MinMaxScaler* and notebook 2, section *Split first, as Lesson 1 insisted*.

In [5]:
train = np.array([[20.0], [40.0], [60.0], [80.0]])   # the four training charges
test = np.array([[100.0]])                             # one new customer

### Your prediction, before you run the next cell

The scaler is fitted on the four training charges (mean 50, standard
deviation 22.36). What does `transform` give for the new customer's 100 euros? And what
would `fit_transform` on the test row alone give?

In [6]:
scaler = StandardScaler()
train_scaled = scaler.fit_transform(train)           # fit + transform on the training rows
print("training rows, scaled:", train_scaled.ravel())
print("same as fit, then transform?",
      np.allclose(train_scaled, StandardScaler().fit(train).transform(train)))

print()
print("RIGHT  transform(test) with the training numbers:", scaler.transform(test).ravel())
print("WRONG  fit_transform(test), learned from the test row:", StandardScaler().fit_transform(test).ravel())
both = np.vstack([train, test])
print("WRONG  fitted on all five rows, then test:", StandardScaler().fit(both).transform(test).ravel().round(3))

training rows, scaled: [-1.342 -0.447  0.447  1.342]
same as fit, then transform? True

RIGHT  transform(test) with the training numbers: [2.236]
WRONG  fit_transform(test), learned from the test row: [0.]
WRONG  fitted on all five rows, then test: [1.414]


**What to look at.** The right answer is (100 - 50) / 22.36 = **2.236**: the new
customer pays more than two standard deviations above the training average, which is
true. Refitting on the test row alone gives **0**: the mean of one value is that value,
so every test customer would look perfectly average. Fitting on all five rows gives
1.414: the test customer pulled the mean up to 60 and the standard deviation up to
28.28, so it looks less unusual than it is. That is leakage in its smallest form: the
test row helped set the ruler it is then measured with.

**Try changing.** make the test set two customers, 100 and 10, and compare
`transform` against `fit_transform` on them.

## 3. `predict` gives a label, `predict_proba` gives the probabilities behind it

A classifier can answer in two ways. `predict` returns one label per row.
`predict_proba` returns one probability per class per row, with columns in the order
of the attribute `classes_`. For two classes, `predict` is simply "is the probability
of the second class at least 0.5?".

*From:* lesson 1, slide *Behind every label, a probability* and notebook 2, section *The threshold nobody chose*.

In [7]:
# Eight students: hours studied, and whether they passed (1) or not (0).
hours = np.array([[1.0], [2.0], [3.0], [4.0], [5.0], [6.0], [7.0], [8.0]])
passed = np.array([0, 0, 0, 1, 0, 1, 1, 1])
model = LogisticRegression().fit(hours, passed)
print("classes_ =", model.classes_)

classes_ = [0 1]


### Your prediction, before you run the next cell

The student who studied 4 hours passed. For a new student who also studied 4
hours, will `predict` say 0 or 1? Will the probability of passing be close to 0.5 or
far from it?

In [8]:
new = np.array([[1.0], [4.0], [5.0], [8.0]])
proba = model.predict_proba(new)
print("hours | P(fail) P(pass) | predict | P(pass) >= 0.5")
for h, (p0, p1), label in zip(new.ravel(), proba, model.predict(new)):
    print(f"{h:5} |  {p0:.3f}   {p1:.3f}  |    {label}    | {p1 >= 0.5}")
print("each row of predict_proba sums to 1:", proba.sum(axis=1))

hours | P(fail) P(pass) | predict | P(pass) >= 0.5
  1.0 |  0.954   0.046  |    0    | False
  4.0 |  0.607   0.393  |    0    | False
  5.0 |  0.393   0.607  |    1    | True
  8.0 |  0.046   0.954  |    1    | True
each row of predict_proba sums to 1: [1. 1. 1. 1.]


**What to look at.** Each row of `predict_proba` has two columns, failing then
passing, because `classes_` is `[0 1]`, and they add up to 1. At 4 hours the
model says 0, although the training student with 4 hours passed: a model summarises
the trend of all eight rows and does not memorise each one. Its probability of passing there
is 0.39, not far from 0.5, so the label is a narrow call, and `predict` alone does not show that. The
last column confirms that `predict` is the probability compared with 0.5.

**Try changing.** change `passed` so that the student with 5 hours passed, refit,
and see how the probability at 4.5 hours moves.

## 4. `score` is a metric with a default

Every predictor has a `score(X, y)` method. It predicts on `X` and compares with
`y` using a default metric: **accuracy** for a classifier, the coefficient of
determination **R²** for a regressor. It is a shortcut, not a different measurement.

*From:* lesson 1, slide *Accuracy: the share of right answers*; lesson 3, slide *A cost needs something to be compared against*.

In [9]:
# Classifier: the students of atom 3, and four new ones to test on.
test_hours = np.array([[2.5], [4.0], [5.5], [7.0]])
test_passed = np.array([0, 0, 0, 1])

# Regressor: the three houses of atom 1, and three new houses.
test_area = np.array([[100.0], [150.0], [180.0]])
test_price = np.array([290_000.0, 400_000.0, 470_000.0])

### Your prediction, before you run the next cell

Will `model.score` and `accuracy_score` give the same number? And will
`regression.score` give the R² or the mean squared error?

In [10]:
print("classifier  score:          ", model.score(test_hours, test_passed))
print("            accuracy_score:   ", accuracy_score(test_passed, model.predict(test_hours)))
print("            share right, by hand:", (model.predict(test_hours) == test_passed).mean())
print()
print("regressor   score:", round(regression.score(test_area, test_price), 4))
print("            r2_score:", round(r2_score(test_price, regression.predict(test_area)), 4))

classifier  score:           0.75
            accuracy_score:    0.75
            share right, by hand: 0.75

regressor   score: 0.9777
            r2_score: 0.9777


**What to look at.** Three routes, one number for the classifier, and two routes,
one number for the regressor. The regressor's `score` is R², not an error: higher is
better, 1 is perfect. When you need anything other than the default metric (recall,
the area under the curve, root mean squared error), call the metric function from
`sklearn.metrics` yourself.

**Try changing.** score the classifier on its own training data, `model.score(hours,
passed)`, and ask why that number says little about new students.

## 5. Why arrays, and why `X` has two dimensions

scikit-learn works on NumPy arrays (pandas DataFrames are converted to arrays on
the way in). An array does arithmetic on every element at once, which a Python list
does not. And the library insists that the inputs `X` form a table, shape
`(m, n)`: $m$ rows (examples), $n$ columns (features), even when there is a single
feature. The target `y` is one value per row, shape `(m,)`.

*From:* lesson 1, notebook 2, section *Before a real model: what "fitting" actually does*; lesson 3, slide *θ and X: all parameters, all houses*.

In [11]:
areas_list = [80, 120, 200]
areas = np.array(areas_list)

### Your prediction, before you run the next cell

What does `areas_list * 2` give, and what does `areas * 2` give?

In [12]:
print("list  * 2:", areas_list * 2)
print("array * 2:", areas * 2)
print("array * 2,400 euros per square metre:", areas * 2_400)

list  * 2: [80, 120, 200, 80, 120, 200]
array * 2: [160 240 400]
array * 2,400 euros per square metre: [192000 288000 480000]


**What to look at.** Multiplying a list repeats it; multiplying an array multiplies
every element. Every formula in lessons 1 to 3, a prediction $wx + b$ for all houses at
once or a mean of squared errors, is written as array arithmetic for this reason.

### Your prediction, before you run the next cell

`areas` has shape `(3,)`. Will `LinearRegression().fit(areas, price)` work?

In [13]:
print("areas.shape =", areas.shape)
try:
    LinearRegression().fit(areas, price)
except ValueError as error:
    print("ValueError:", str(error).splitlines()[0])

X = areas.reshape(-1, 1)     # -1 means "as many rows as needed", 1 column
print("X.shape =", X.shape, "  y.shape =", price.shape)
print("fits now, slope =", LinearRegression().fit(X, price).coef_.round(1))

areas.shape = (3,)
ValueError: Expected 2D array, got 1D array instead:
X.shape = (3, 1)   y.shape = (3,)
fits now, slope = [2535.7]


**What to look at.** A one-dimensional array is ambiguous: three examples of one
feature, or one example of three features? scikit-learn refuses to guess and raises
an error. `reshape(-1, 1)` turns it into three rows of one column. With a DataFrame,
`df[["area"]]` (double brackets) gives a table and `df["area"]` gives a single column,
which is why the notebooks select inputs with double brackets.

### Your prediction, before you run the next cell

A scaler is fitted on a DataFrame with named columns. Does `transform`
return a DataFrame or an array?

In [14]:
df = pd.DataFrame({"area": [80, 120, 200], "rooms": [2, 5, 4]})
scaler = StandardScaler().fit(df)
out = scaler.transform(df)
print("type of the output:", type(out).__name__)
print("feature_names_in_: ", scaler.feature_names_in_)
print("with set_output, a DataFrame again:")
print(StandardScaler().set_output(transform="pandas").fit_transform(df).round(3))

type of the output: ndarray
feature_names_in_:  ['area' 'rooms']
with set_output, a DataFrame again:
    area  rooms
0 -1.069 -1.336
1 -0.267  1.069
2  1.336  0.267


**What to look at.** The column names go in and are remembered in
`feature_names_in_`, but what comes out is a plain array: the names are gone unless
you ask for them back with `set_output(transform="pandas")`. That is why lesson 2's
notebook calls `get_feature_names_out()` to label the columns after a
`ColumnTransformer`.

**Try changing.** call `areas.reshape(1, -1)` and look at the shape: it is one row of
three features, the other reading of the ambiguous array.

## 6. `random_state`: the same randomness, every time

Splitting, shuffling and some models draw random numbers. `random_state` fixes the
starting point of that draw, the *seed*, so the same call gives the same result on
every run and on every computer. Without it, every run draws differently.

*From:* lesson 1, slide *One split is one measurement*.

In [15]:
rows = np.arange(10).reshape(-1, 1)    # ten rows, numbered 0 to 9

### Your prediction, before you run the next cell

Two calls with `random_state=0`: same test rows or different ones? And
a call with `random_state=1`?

In [16]:
for seed in (0, 0, 1, 2):
    _, test_rows = train_test_split(rows, test_size=3, random_state=seed)
    print(f"random_state={seed}: test rows {test_rows.ravel()}")

random_state=0: test rows [2 8 4]
random_state=0: test rows [2 8 4]
random_state=1: test rows [2 9 6]
random_state=2: test rows [4 1 5]


**What to look at.** The same seed gives the same three test rows; a different seed
gives a different, equally legitimate split. Leaving `random_state` out means a new
seed on every run, so a notebook would print a different accuracy every time it is
executed. The seed makes a result *reproducible*; it does not make it *right*: lesson
1's slide *One split is one measurement* shows how much the score moves from one seed
to the next.

**Try changing.** remove `random_state` and run the cell three times.

## 7. `Pipeline`: one `fit`, and every step fitted on the right rows

`make_pipeline(step1, step2, model)` chains estimators into one. Calling `fit` on
the pipeline calls `fit_transform` on each transformer in turn and `fit` on the
model at the end. Calling `predict` calls `transform` on each transformer, never
`fit`, and then `predict` on the model. So the rule of atom 2 is applied for you.

*From:* lesson 1, slide *Step 5: pipeline, not two steps*; lesson 2, slide *ColumnTransformer + Pipeline*.

In [17]:
# The eight students of atom 3 for training, the four of atom 4 for testing.
pipe = make_pipeline(StandardScaler(), LogisticRegression())
pipe.fit(hours, passed)
pipe

Pipeline(steps=[('standardscaler', StandardScaler()),
                ('logisticregression', LogisticRegression())])

### Your prediction, before you run the next cell

What is the scaler's `mean_` inside the pipeline: the mean of the
training hours, or of the training and test hours together?

In [18]:
inner_scaler = pipe[0]                         # steps can be reached by position
print("scaler mean_ in the pipeline:", inner_scaler.mean_, "  training mean:", hours.mean())

by_hand = pipe[-1].predict(inner_scaler.transform(test_hours))   # what predict does inside
print("pipe.predict(test):", pipe.predict(test_hours))
print("by hand:           ", by_hand)
print("step names:", list(pipe.named_steps))

scaler mean_ in the pipeline: [4.5]   training mean: 4.5
pipe.predict(test): [0 0 1 1]
by hand:            [0 0 1 1]
step names: ['standardscaler', 'logisticregression']


**What to look at.** The scaler inside the pipeline learned 4.5, the mean of the
eight training hours, and the test rows never reached its `fit`. The prediction done
by hand, transform with the stored numbers and then predict, gives the same labels as
`pipe.predict`. `make_pipeline` names each step after its class in lower case, which
is how `named_steps` reaches them.

**Try changing.** add `SimpleImputer()` as the first step and put a missing value in
`test_hours`: the pipeline still predicts, with the hole filled by the training mean.

## 8. The contract, in one table

Which methods does each object have? The code asks the objects themselves, so the
table cannot drift from the library.

*From:* every notebook of lessons 1 to 3.

In [19]:
objects = {
    "StandardScaler": StandardScaler(),
    "SimpleImputer": SimpleImputer(),
    "OneHotEncoder": OneHotEncoder(),
    "LinearRegression": LinearRegression(),
    "LogisticRegression": LogisticRegression(),
    "DummyClassifier": DummyClassifier(),
    "scaler + logistic pipeline": make_pipeline(StandardScaler(), LogisticRegression()),
}
methods = ["fit", "transform", "fit_transform", "predict", "predict_proba", "score"]
table = pd.DataFrame({name: ["yes" if hasattr(obj, m) else "" for m in methods]
                      for name, obj in objects.items()}, index=methods).T
table

,fit,transform,fit_transform,predict,predict_proba,score
StandardScaler,yes,yes,yes,,,
SimpleImputer,yes,yes,yes,,,
OneHotEncoder,yes,yes,yes,,,
LinearRegression,yes,,,yes,,yes
LogisticRegression,yes,,,yes,yes,yes
DummyClassifier,yes,,,yes,yes,yes
scaler + logistic pipeline,yes,,,yes,yes,yes


**What to look at.** Transformers have `transform` and `fit_transform`; predictors
have `predict` and `score`; only classifiers have `predict_proba`. A pipeline has the
methods of its *last* step, so a pipeline ending in a classifier predicts but does not
transform. The rule for each method is summarised below.

| method | learns from the data? | returns | call it on |
|---|---|---|---|
| `fit(X, y)` | yes, stores attributes ending in `_` | the estimator itself | training rows only |
| `transform(X)` | no, uses what `fit` stored | the changed data | training and test rows |
| `fit_transform(X, y)` | yes, then applies it | the changed data | training rows only |
| `predict(X)` | no | one label or value per row | any rows |
| `predict_proba(X)` | no | one probability per class per row | any rows |
| `score(X, y)` | no | accuracy (classifier) or R² (regressor) | the rows you want to measure |